# 01 — Exploración inicial de los datos ESP

## Objetivo pedagógico

Reconstruir, celda por celda, la primera lectura del conjunto experimental de transiciones de patrón de flujo en una bomba electrosumergible con gas.

El cuaderno enseña a localizar los archivos, a reconocer que un Excel de laboratorio no llega con una tabla plana y a contar pares numéricos sin convertirlos todavía en una conclusión física.

## Alcance

Las actividades 2.3 a 2.10 examinan solo `mapping.xlsx`, hoja `50psig`. Las otras hojas comparten los nombres `50psig`, `100psig` y `150psig`, pero su tamaño es distinto. Sus posiciones de columna no se copian desde esta hoja.

Un par numérico completo significa que caudal y `DP2-3` se pueden leer como números en la misma fila. Ese conteo no certifica una medición físicamente validada ni un experimento independiente.

Después de las verificaciones, la actividad 2.11 cierra el diccionario de variables. No cambia los conteos de las actividades anteriores.

## Actividad 2.1 — Localización de archivos

El proyecto se busca desde el directorio de trabajo actual, subiendo por los directorios padres hasta encontrar las dos copias de trabajo. Así el cuaderno funciona tanto si el kernel arranca en `notebooks/` como si arranca en la raíz del laboratorio.

In [1]:
from pathlib import Path

import pandas as pd


def encontrar_raiz_laboratorio() -> Path:
    """Sube desde el directorio de trabajo hasta la raíz de ESP-DataScience."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for candidato in candidatos:
        mapping = candidato / "data" / "raw" / "mapping.xlsx"
        surging = candidato / "data" / "raw" / "surging.xlsx"
        if mapping.is_file() and surging.is_file():
            return candidato
    raise FileNotFoundError(
        "No se encontró data/raw/mapping.xlsx junto a data/raw/surging.xlsx."
    )


raiz = encontrar_raiz_laboratorio()
ruta_mapping = raiz / "data" / "raw" / "mapping.xlsx"
ruta_surging = raiz / "data" / "raw" / "surging.xlsx"

print("Raíz del laboratorio localizada:", raiz.name)
print("mapping.xlsx existe:", ruta_mapping.is_file())
print("surging.xlsx existe:", ruta_surging.is_file())


Raíz del laboratorio localizada: ESP-DataScience
mapping.xlsx existe: True
surging.xlsx existe: True


## Actividad 2.2 — Identificación de hojas

En el análisis anterior ambos libros contenían las hojas `50psig`, `100psig` y `150psig`. Aquí se comprueba esa lista y, además, se imprime el tamaño de cada hoja.

El tamaño sirve como advertencia: coincidir en el nombre de la hoja no autoriza a reutilizar las columnas descubiertas en Mapping/`50psig`.

In [2]:
hojas_esperadas = ["50psig", "100psig", "150psig"]
archivos = {"mapping": ruta_mapping, "surging": ruta_surging}

for nombre, ruta in archivos.items():
    with pd.ExcelFile(ruta) as libro:
        print(f"{nombre}: {libro.sheet_names}")
        faltan = [hoja for hoja in hojas_esperadas if hoja not in libro.sheet_names]
        if faltan:
            raise AssertionError(f"{nombre} no contiene las hojas {faltan}.")
        for hoja in libro.sheet_names:
            temporal = pd.read_excel(libro, sheet_name=hoja, header=None)
            print(f"  {hoja}: {temporal.shape[0]} filas x {temporal.shape[1]} columnas")

mapping: ['50psig', '100psig', '150psig']
  50psig: 50 filas x 18 columnas
  100psig: 52 filas x 18 columnas
  150psig: 55 filas x 18 columnas
surging: ['50psig', '100psig', '150psig']
  50psig: 50 filas x 9 columnas
  100psig: 61 filas x 9 columnas
  150psig: 56 filas x 9 columnas


## Actividad 2.3 — Inspección inicial

Se carga únicamente la hoja `50psig` de mapping, con `header=None`, para no convertir la primera fila de condiciones en nombres de columna.

En el laboratorio anterior esta lectura tenía 50 filas y 18 columnas. La celda lo comprueba y muestra las primeras 15 filas.

In [3]:
df = pd.read_excel(ruta_mapping, sheet_name="50psig", header=None)

print("Dimensiones:", df.shape)
if df.shape != (50, 18):
    raise AssertionError(
        f"Se esperaban 50 filas y 18 columnas; la hoja mide {df.shape}."
    )

df.head(15)

Dimensiones: (50, 18)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,NaN,Single Phase,NaN,NaN,Qgd=0.005,NaN,NaN,Qgd=0.01,NaN,NaN,Qgd=0.015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rotary speed,Flow rate,DP2-3,NaN,Flow rate,DP2-3,NaN,Flow rate,DP2-3,NaN,Flow rate,DP2-3,NaN,NaN,NaN,NaN,NaN,NaN
2,1800,424.621104,8.38933,NaN,2351.750216,0.395417,NaN,2316.374106,0.420945,NaN,2278.317547,0.46786,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,606.380015,8.242284,NaN,2251.875003,1.148531,NaN,2195.842004,1.267744,NaN,2144.048733,1.281426,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,900.373188,7.771301,NaN,2036.58316,2.585132,NaN,2027.375352,2.463128,NaN,1983.697641,1.887018,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,1137.588486,7.17206,NaN,1911.382242,3.379172,NaN,1903.920476,3.140544,NaN,1945.391096,2.235566,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,1326.837735,6.571055,NaN,1797.217779,3.985228,NaN,1802.931357,3.784445,NaN,1820.755212,2.816368,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,1522.941113,5.624306,NaN,1716.175542,4.5443,NaN,1695.581354,4.331602,NaN,1710.392528,3.555644,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,1732.732837,4.545451,NaN,1599.303894,5.107235,NaN,1607.637824,4.670273,NaN,1601.542689,3.853203,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,1918.125654,3.499286,NaN,1512.631136,5.550768,NaN,1506.01773,5.281399,NaN,1497.705274,4.322578,NaN,NaN,NaN,NaN,NaN,NaN


## Actividad 2.4 — Inventario de columnas

La fila 0 guarda la condición experimental de cada bloque de columnas. La fila 1 guarda el nombre de la variable.

El inventario lista todas las columnas, también aquellas cuyo encabezado está vacío. Un encabezado vacío no se interpreta, en esta actividad, como una columna sin datos.

In [4]:
inventario_encabezados = pd.DataFrame(
    {
        "indice": [int(columna) for columna in df.columns],
        "condicion_experimental": [df.iloc[0, columna] for columna in df.columns],
        "nombre_variable": [df.iloc[1, columna] for columna in df.columns],
    }
)

inventario_encabezados

,indice,condicion_experimental,nombre_variable
0,0,NaN,rotary speed
1,1,Single Phase,Flow rate
2,2,NaN,DP2-3
3,3,NaN,NaN
4,4,Qgd=0.005,Flow rate
5,5,NaN,DP2-3
6,6,NaN,NaN
7,7,Qgd=0.01,Flow rate
8,8,NaN,DP2-3
9,9,NaN,NaN


## Actividad 2.5 — Perfilamiento de columnas

Se cuenta cuántas celdas de cada columna contienen algún valor. Una columna completamente vacía es la que tiene cero celdas con información.

En el laboratorio anterior esas columnas fueron `[3, 6, 9, 12, 15]`. Se comprueba el resultado y no se elimina ninguna columna.

In [5]:
perfil = pd.DataFrame(
    {
        "indice": [int(columna) for columna in df.columns],
        "celdas_con_informacion": [int(df[columna].notna().sum()) for columna in df.columns],
    }
)

columnas_vacias = perfil.loc[
    perfil["celdas_con_informacion"] == 0, "indice"
].tolist()

print("Columnas completamente vacías:", columnas_vacias)
if columnas_vacias != [3, 6, 9, 12, 15]:
    raise AssertionError(
        "Las columnas vacías no coinciden con [3, 6, 9, 12, 15]: "
        f"{columnas_vacias}"
    )

perfil

Columnas completamente vacías: [3, 6, 9, 12, 15]


,indice,celdas_con_informacion
0,0,4
1,1,35
2,2,33
3,3,0
4,4,49
5,5,47
6,6,0
7,7,43
8,8,41
9,9,0


## Actividad 2.6 — Investigación de bloques

Las columnas `[0, 13, 14, 16, 17]` se revisan enteras. En la parte alta de la hoja varias de ellas no tienen encabezado; más abajo aparecen rótulos intermedios y números.

La columna 0 registra dos velocidades numéricas, 1800 y 3500. Esta actividad las identifica. La actividad 2.11, ya cerrada, fija la unidad en rpm.

In [6]:
columnas_de_bloque = [0, 13, 14, 16, 17]

for columna in columnas_de_bloque:
    print(f"--- Columna {columna} ---")
    for fila, valor in df[columna].items():
        if pd.notna(valor):
            es_numero = isinstance(valor, (int, float)) and not isinstance(valor, bool)
            clase = "numerico" if es_numero else "encabezado_o_texto"
            print(f"  fila {int(fila):>2}: {valor!r} ({clase})")

velocidades = []
for valor in df[0].dropna():
    if isinstance(valor, (int, float)) and not isinstance(valor, bool):
        velocidades.append(int(valor))

print("Velocidades numéricas en la columna 0:", velocidades)
if velocidades != [1800, 3500]:
    raise AssertionError(
        f"Se esperaban las velocidades [1800, 3500]; se leyeron {velocidades}."
    )
print("Unidad de la velocidad: rpm. La actividad 2.11 la toma de la nomenclatura.")

--- Columna 0 ---
  fila  1: 'rotary speed' (encabezado_o_texto)
  fila  2: 1800 (numerico)
  fila 23: 'rotary speed' (encabezado_o_texto)
  fila 24: 3500 (numerico)
--- Columna 13 ---
  fila 22: 'Qgd=0.04' (encabezado_o_texto)
  fila 23: 'Flow rate' (encabezado_o_texto)
  fila 24: 4263.686967969283 (numerico)
  fila 25: 4179.539194688038 (numerico)
  fila 26: 4078.9636376372646 (numerico)
  fila 27: 3936.6261478298297 (numerico)
  fila 28: 3702.050330523026 (numerico)
  fila 29: 3506.26723941042 (numerico)
  fila 30: 3299.3557486267823 (numerico)
  fila 31: 3099.80106748847 (numerico)
  fila 32: 2789.1253097641315 (numerico)
  fila 33: 2692.120871252887 (numerico)
  fila 34: 2639.1537351848983 (numerico)
  fila 35: 2447.5735824914177 (numerico)
  fila 36: 2416.234361482914 (numerico)
  fila 37: 2357.151253318793 (numerico)
  fila 38: 2260.6833344366137 (numerico)
  fila 39: 2079.0740109891026 (numerico)
  fila 40: 1930.5236439519351 (numerico)
  fila 41: 1993.232852121748 (numerico)
 

## Actividad 2.7 — Estructura vertical

Los rótulos de condición no ocupan una sola franja. Hay un bloque que empieza en la fila 0 y otro que empieza en la fila 22. Cada bloque trae su propia fila de nombres de variable en la fila siguiente.

Esta separación explica por qué `Qgd=0.04` y `Qgd=0.05` no aparecieron en el inventario de las filas 0 y 1.

In [7]:
for fila_condicion in (0, 22):
    print(f"Fila {fila_condicion} — condiciones")
    for columna in df.columns:
        valor = df.iloc[fila_condicion, columna]
        if pd.notna(valor):
            print(f"  columna {int(columna)}: {valor}")
    fila_variable = fila_condicion + 1
    print(f"Fila {fila_variable} — nombres de variable")
    for columna in df.columns:
        valor = df.iloc[fila_variable, columna]
        if pd.notna(valor):
            print(f"  columna {int(columna)}: {valor}")
    print()

Fila 0 — condiciones
  columna 1: Single Phase
  columna 4: Qgd=0.005
  columna 7: Qgd=0.01
  columna 10: Qgd=0.015
Fila 1 — nombres de variable
  columna 0: rotary speed
  columna 1: Flow rate
  columna 2: DP2-3
  columna 4: Flow rate
  columna 5: DP2-3
  columna 7: Flow rate
  columna 8: DP2-3
  columna 10: Flow rate
  columna 11: DP2-3

Fila 22 — condiciones
  columna 1: Single Phase
  columna 4: Qgd=0.01
  columna 7: Qgd=0.02
  columna 10: Qgd=0.03
  columna 13: Qgd=0.04
  columna 16: Qgd=0.05
Fila 23 — nombres de variable
  columna 0: rotary speed
  columna 1: Flow rate
  columna 2: DP2-3
  columna 4: Flow rate
  columna 5: DP2-3
  columna 7: Flow rate
  columna 8: DP2-3
  columna 10: Flow rate
  columna 11: DP2-3
  columna 13: Flow rate
  columna 14: DP2-3
  columna 16: Flow rate
  columna 17: DP2-3



## Actividad 2.8 — Inventario estructurado

`df_inventario` resume la hoja `50psig` de mapping. La velocidad de cada bloque se lee de la primera fila de datos de la columna 0: la fila 2 para el primer bloque y la fila 24 para el segundo.

La columna de la condición es la columna de caudal (`Flow rate`). `DP2-3` se toma de la columna inmediata siguiente, después de comprobar que ese es el nombre escrito en la fila de variables.

El inventario resultante debe contener diez condiciones: cuatro asociadas a 1800 y seis asociadas a 3500. Estas filas y columnas pertenecen a esta hoja; no se aplican al resto del libro ni a surging.

In [8]:
bloques = [
    {
        "fila_condicion": 0,
        "fila_variable": 1,
        "fila_datos_inicio": 2,
        "fila_datos_fin": 22,
    },
    {
        "fila_condicion": 22,
        "fila_variable": 23,
        "fila_datos_inicio": 24,
        "fila_datos_fin": len(df),
    },
]

registros = []
for bloque in bloques:
    velocidad = int(df.iloc[bloque["fila_datos_inicio"], 0])
    for columna in df.columns:
        condicion = df.iloc[bloque["fila_condicion"], columna]
        if pd.isna(condicion):
            continue
        columna_caudal = int(columna)
        columna_dp = columna_caudal + 1
        nombre_caudal = df.iloc[bloque["fila_variable"], columna_caudal]
        nombre_dp = df.iloc[bloque["fila_variable"], columna_dp]
        if nombre_caudal != "Flow rate" or nombre_dp != "DP2-3":
            raise AssertionError(
                "El par de columnas no corresponde a Flow rate y DP2-3: "
                f"velocidad {velocidad}, condición {condicion!r}, "
                f"columnas {columna_caudal} y {columna_dp} "
                f"({nombre_caudal!r}, {nombre_dp!r})."
            )
        registros.append(
            {
                "hoja": "50psig",
                "velocidad": velocidad,
                "condicion": condicion,
                "columna_caudal": columna_caudal,
                "columna_dp": columna_dp,
            }
        )

df_inventario = pd.DataFrame(registros)

condiciones_esperadas = [
    (1800, "Single Phase"),
    (1800, "Qgd=0.005"),
    (1800, "Qgd=0.01"),
    (1800, "Qgd=0.015"),
    (3500, "Single Phase"),
    (3500, "Qgd=0.01"),
    (3500, "Qgd=0.02"),
    (3500, "Qgd=0.03"),
    (3500, "Qgd=0.04"),
    (3500, "Qgd=0.05"),
]
condiciones_leidas = list(
    zip(df_inventario["velocidad"].tolist(), df_inventario["condicion"].tolist())
)
if condiciones_leidas != condiciones_esperadas:
    raise AssertionError(
        "El inventario no reproduce las diez condiciones de Mapping/50psig.\n"
        f"Leídas: {condiciones_leidas}"
    )

df_inventario

,hoja,velocidad,condicion,columna_caudal,columna_dp
0,50psig,1800,Single Phase,1,2
1,50psig,1800,Qgd=0.005,4,5
2,50psig,1800,Qgd=0.01,7,8
3,50psig,1800,Qgd=0.015,10,11
4,50psig,3500,Single Phase,1,2
5,50psig,3500,Qgd=0.01,4,5
6,50psig,3500,Qgd=0.02,7,8
7,50psig,3500,Qgd=0.03,10,11
8,50psig,3500,Qgd=0.04,13,14
9,50psig,3500,Qgd=0.05,16,17


## Actividad 2.9 — Conteo de observaciones

Para cada condición se recorta el bloque vertical ya delimitado:

- velocidad 1800: desde la fila 2 hasta antes de la fila 22;
- velocidad 3500: desde la fila 24 hasta el final.

Dentro de ese recorte, `pd.to_numeric(errors="coerce")` convierte el texto no numérico en valor ausente. El conteo conserva solo las filas donde caudal y `DP2-3` son numéricos a la vez. Las filas incompletas se excluyen de la cuenta y permanecen en el Excel.

In [9]:
rangos_por_velocidad = {
    1800: (2, 22),
    3500: (24, len(df)),
}

conteos = []
for registro in df_inventario.itertuples(index=False):
    inicio, fin = rangos_por_velocidad[int(registro.velocidad)]
    bloque = df.iloc[inicio:fin]
    caudal = pd.to_numeric(bloque[registro.columna_caudal], errors="coerce")
    dp = pd.to_numeric(bloque[registro.columna_dp], errors="coerce")
    pares = int((caudal.notna() & dp.notna()).sum())
    conteos.append(
        {
            "velocidad": int(registro.velocidad),
            "condicion": registro.condicion,
            "pares_completos": pares,
        }
    )

df_conteos = pd.DataFrame(conteos)
df_conteos

,velocidad,condicion,pares_completos
0,1800,Single Phase,10
1,1800,Qgd=0.005,19
2,1800,Qgd=0.01,18
3,1800,Qgd=0.015,15
4,3500,Single Phase,21
5,3500,Qgd=0.01,26
6,3500,Qgd=0.02,21
7,3500,Qgd=0.03,21
8,3500,Qgd=0.04,20
9,3500,Qgd=0.05,15


## Actividad 2.10 — Verificaciones

Se contrastan cuatro totales del análisis anterior:

- 186 pares completos;
- 10 condiciones inventariadas;
- 62 pares a velocidad 1800;
- 124 pares a velocidad 3500.

Si alguno no coincide, la celda se detiene y muestra el valor leído, para investigar la diferencia antes de modificar el criterio o la cifra esperada.

Estas cifras cuentan pares numéricos completos en la hoja. No cuentan mediciones físicamente validadas ni experimentos independientes.

In [10]:
total_pares = int(df_conteos["pares_completos"].sum())
condiciones = int(len(df_inventario))
pares_1800 = int(
    df_conteos.loc[df_conteos["velocidad"] == 1800, "pares_completos"].sum()
)
pares_3500 = int(
    df_conteos.loc[df_conteos["velocidad"] == 3500, "pares_completos"].sum()
)

verificaciones = {
    "pares_completos": (total_pares, 186),
    "condiciones": (condiciones, 10),
    "pares_1800": (pares_1800, 62),
    "pares_3500": (pares_3500, 124),
}

for nombre, (obtenido, esperado) in verificaciones.items():
    print(f"{nombre}: obtenido {obtenido}, esperado {esperado}")
    if obtenido != esperado:
        raise AssertionError(
            f"{nombre} no coincide: obtenido {obtenido}, esperado {esperado}."
        )

print("Verificaciones 186 / 10 / 62 / 124: coinciden.")
print(
    "Lectura del resultado: pares numéricos completos en Mapping/50psig, "
    "no mediciones físicamente validadas ni experimentos independientes."
)

pares_completos: obtenido 186, esperado 186
condiciones: obtenido 10, esperado 10
pares_1800: obtenido 62, esperado 62
pares_3500: obtenido 124, esperado 124
Verificaciones 186 / 10 / 62 / 124: coinciden.
Lectura del resultado: pares numéricos completos en Mapping/50psig, no mediciones físicamente validadas ni experimentos independientes.


## Actividad 2.11 — Diccionario científico de variables

Esta actividad continúa el cuaderno. Las fuentes son el artículo de Zhu y colaboradores (2019), el artículo experimental de 2018 sobre surfactante (Zhu et al., *Experimental Thermal and Fluid Science*) y la tesis doctoral de Jianjun Zhu (University of Tulsa, 2017). El diccionario completo está en `docs/diccionario_variables.md`.

La tabla de abajo resume la lectura de trabajo. Una certeza **media** significa que el texto define la magnitud y el Excel usa otro rótulo: eso es una interpretación, no una cita literal.

- Qmax de la TE-2700 a 3500 rpm es 4900 bpd. QBEP a esa velocidad es 2700 bpd.
- `Flow rate` se lee como caudal de líquido en bpd.
- `DP2-3` se lee como el incremento de presión de la etapa 3, en psi. El nombre no está en la tesis.
- `GVF0` se lee como la fracción de gas de admisión en porcentaje. El sufijo 0 no está definido.
- En mapping, la franja de Qgd = 0,005, 0,01 y 0,015 es de 1800 rpm. La de Qgd = 0,01 a 0,05 es de 3500 rpm. Eso asigna la velocidad de las hojas `100psig` y `150psig`, cuyas celdas están vacías.


In [11]:
articulo = raiz / "docs" / "esp_gassy_flow_paper.pdf"
tesis = raiz / "ProjectESP" / "Papers" / "Thesis" / "Two-phase" / "Zhu 2017 (PhD thesis).pdf"
surfactante = raiz / "ProjectESP" / "Papers" / "Zhu 2018b.pdf"
for fuente in (articulo, tesis, surfactante):
    if not fuente.is_file():
        raise FileNotFoundError("Falta una fuente del diccionario.")
print("Fuentes locales presentes: artículo de 2019, tesis de 2017 y artículo de 2018b.")

df_diccionario = pd.DataFrame(
    [
        {
            "simbolo": "Qgd",
            "definicion": "QG / Qmax",
            "unidad": "Adimensional. Qmax a 3500 rpm: 4900 bpd",
            "columna_o_rotulo": "Rótulo de condición en mapping",
            "certeza": "Alta",
        },
        {
            "simbolo": "QL",
            "definicion": "Caudal de líquido",
            "unidad": "bpd en la tesis de estos ensayos",
            "columna_o_rotulo": "Flow rate",
            "certeza": "Alta como interpretación",
        },
        {
            "simbolo": "QG",
            "definicion": "Caudal de gas de admisión",
            "unidad": "Entra en Qgd. No hay columna QG",
            "columna_o_rotulo": "Rótulo Qgd en mapping",
            "certeza": "Alta para la definición",
        },
        {
            "simbolo": "DP2-3",
            "definicion": "Incremento de presión de la etapa 3",
            "unidad": "psi en la tesis. El mejor punto es 22,5 psi",
            "columna_o_rotulo": "DP2-3",
            "certeza": "Media. El rótulo no está en la tesis",
        },
        {
            "simbolo": "N",
            "definicion": "Velocidad de rotación",
            "unidad": "rpm",
            "columna_o_rotulo": "1800 y 3500. En mapping 100 y 150 psig, por el diseño",
            "certeza": "Alta",
        },
        {
            "simbolo": "Psep",
            "definicion": "Presión del separador",
            "unidad": "psig",
            "columna_o_rotulo": "Nombre de la hoja",
            "certeza": "Alta",
        },
        {
            "simbolo": "GVF0",
            "definicion": "Fracción de gas de admisión, QG / (QG + QL)",
            "unidad": "Porcentaje, como en las figuras",
            "columna_o_rotulo": "GVF0 en surging",
            "certeza": "Media. El sufijo 0 no está definido",
        },
        {
            "simbolo": "QBEP",
            "definicion": "Caudal en el mejor punto de eficiencia",
            "unidad": "2700 bpd a 3500 rpm",
            "columna_o_rotulo": "0.75Qbep, Qbep y 1.25Qbep",
            "certeza": "Alta",
        },
    ]
)
df_diccionario


Fuentes locales presentes: artículo de 2019, tesis de 2017 y artículo de 2018b.


,simbolo,definicion,unidad,columna_o_rotulo,certeza
0,Qgd,QG / Qmax,Adimensional. Qmax a 3500 rpm: 4900 bpd,Rótulo de condición en mapping,Alta
1,QL,Caudal de líquido,bpd en la tesis de estos ensayos,Flow rate,Alta como interpretación
2,QG,Caudal de gas de admisión,Entra en Qgd. No hay columna QG,Rótulo Qgd en mapping,Alta para la definición
3,DP2-3,Incremento de presión de la etapa 3,"psi en la tesis. El mejor punto es 22,5 psi",DP2-3,Media. El rótulo no está en la tesis
4,N,Velocidad de rotación,rpm,"1800 y 3500. En mapping 100 y 150 psig, por el...",Alta
5,Psep,Presión del separador,psig,Nombre de la hoja,Alta
6,GVF0,"Fracción de gas de admisión, QG / (QG + QL)","Porcentaje, como en las figuras",GVF0 en surging,Media. El sufijo 0 no está definido
7,QBEP,Caudal en el mejor punto de eficiencia,2700 bpd a 3500 rpm,"0.75Qbep, Qbep y 1.25Qbep",Alta


## Actividad 2.11 — Enlace con el inventario de Mapping/50psig

Cada condición del inventario ya tiene velocidad escrita en la hoja. El caudal es QL en bpd y `DP2-3` es el incremento de la etapa 3 en psi. Esa lectura de presión es la interpretación de certeza media del diccionario.


In [12]:
def certeza_de_condicion(condicion):
    if condicion == "Single Phase":
        return "Curva de agua, serie sin IPA"
    if str(condicion).startswith("Qgd="):
        return "Qgd = QG / Qmax, adimensional"
    return "Sin enlace"

df_enlace = df_inventario.copy()
df_enlace["caudal"] = "QL en bpd"
df_enlace["incremento_etapa"] = "Etapa 3, en psi"
df_enlace["enlace_de_la_condicion"] = df_enlace["condicion"].map(certeza_de_condicion)
interpretaciones = [
    "DP2-3 es el nombre del Excel, no una cita de la tesis.",
    "GVF0 se lee en porcentaje; el sufijo 0 no está definido.",
    "En Mapping/100psig y Mapping/150psig la velocidad no está en la celda. La asigna el diseño: Qgd 0,005 a 0,015 a 1800 rpm, y Qgd 0,01 a 0,05 a 3500 rpm.",
]
print("Interpretaciones que ninguna fuente escribe de forma literal:")
for frase in interpretaciones:
    print("-", frase)
df_enlace


Interpretaciones que ninguna fuente escribe de forma literal:
- DP2-3 es el nombre del Excel, no una cita de la tesis.
- GVF0 se lee en porcentaje; el sufijo 0 no está definido.
- En Mapping/100psig y Mapping/150psig la velocidad no está en la celda. La asigna el diseño: Qgd 0,005 a 0,015 a 1800 rpm, y Qgd 0,01 a 0,05 a 3500 rpm.


,hoja,velocidad,condicion,columna_caudal,columna_dp,caudal,incremento_etapa,enlace_de_la_condicion
0,50psig,1800,Single Phase,1,2,QL en bpd,"Etapa 3, en psi","Curva de agua, serie sin IPA"
1,50psig,1800,Qgd=0.005,4,5,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"
2,50psig,1800,Qgd=0.01,7,8,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"
3,50psig,1800,Qgd=0.015,10,11,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"
4,50psig,3500,Single Phase,1,2,QL en bpd,"Etapa 3, en psi","Curva de agua, serie sin IPA"
5,50psig,3500,Qgd=0.01,4,5,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"
6,50psig,3500,Qgd=0.02,7,8,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"
7,50psig,3500,Qgd=0.03,10,11,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"
8,50psig,3500,Qgd=0.04,13,14,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"
9,50psig,3500,Qgd=0.05,16,17,QL en bpd,"Etapa 3, en psi","Qgd = QG / Qmax, adimensional"


## Qué queda para la actividad 2.13

La actividad 2.12 construye esas dos tablas en `notebooks/02_normalizacion.ipynb`. No se apilan, no se modifican los Excel de `data/raw` y no se entrena ningún modelo. El siguiente paso, todavía sin hacer, es la actividad 2.13. El plan está en `docs/plan_implementacion.md`.
